# Incompressible Fluid Simulation via Fourier Methods

Simulating the motion of a viscous incompressible fluid requires solving the **Navier–Stokes equations**:
$$
\partial_t u + (u \cdot \nabla) u = -\nabla p + \nu \Delta u, \qquad \nabla \cdot u = 0,
$$
where $u : [0,1]^2 \to \mathbb{R}^2$ is the velocity field, $p$ is pressure, and $\nu$ is viscosity. The divergence-free constraint $\nabla \cdot u = 0$ (incompressibility) makes pressure a Lagrange multiplier.

## Fourier spectral method

On a periodic domain $[0,1]^2$, the Fourier transform converts differential operators to multiplication:
$$
\widehat{\nabla f}(k) = 2\pi i\, k\, \hat{f}(k), \qquad \widehat{\Delta f}(k) = -4\pi^2 \|k\|^2\, \hat{f}(k).
$$
This allows:
- **Fast convolutions** via FFT (instead of direct spatial convolution).
- **Exact Laplacian inversion** $\Delta^{-1}$ via division by $-4\pi^2 \|k\|^2$.
- **Helmholtz–Hodge projection** onto divergence-free fields.

## Helmholtz–Hodge decomposition

Any vector field $v$ on a periodic domain decomposes uniquely into a **divergence-free** part and a **curl-free** (gradient) part:
$$
v = u + \nabla p, \qquad \nabla \cdot u = 0.
$$
Taking the divergence: $\Delta p = \nabla \cdot v$, so $p = \Delta^{-1}(\nabla \cdot v)$. The projection $P_{\text{div-free}}(v) = v - \nabla(\Delta^{-1}(\nabla \cdot v))$ is exact in Fourier space:
$$
\hat{u}(k) = \hat{v}(k) - k \frac{k \cdot \hat{v}(k)}{\|k\|^2}.
$$

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

plt.rcParams['figure.dpi'] = 120

n = 64  # grid size (fast for notebook)

# Fourier frequencies
omega = np.fft.fftfreq(n) * n   # integer frequencies 0..n/2,-n/2+1..-1
OX, OY = np.meshgrid(omega, omega)  # (n,n)
R2 = OX**2 + OY**2
R2[0, 0] = 1.0   # avoid division by zero

def gauss_filter(f, sigma):
    F = np.fft.fft2(f)
    G = np.exp(-(OX**2 + OY**2) * sigma**2 / 2)
    return np.fft.ifft2(F * G).real

def grad(f):
    F = np.fft.fft2(f)
    gx = np.fft.ifft2(1j * OX * F).real
    gy = np.fft.ifft2(1j * OY * F).real
    return gx, gy

def div(vx, vy):
    return (np.fft.ifft2(1j * OX * np.fft.fft2(vx))
          + np.fft.ifft2(1j * OY * np.fft.fft2(vy))).real

def laplacian_inv(f):
    F = np.fft.fft2(f)
    F[0, 0] = 0
    return np.fft.ifft2(F / (-R2)).real

def proj_incompressible(vx, vy):
    """Project onto divergence-free fields (Helmholtz-Hodge)."""
    d = div(vx, vy)
    p = laplacian_inv(d)
    px, py = grad(p)
    return vx - px, vy - py

print('Fluid solver ready.')

## Helmholtz–Hodge decomposition

We decompose a random smooth vector field into its divergence-free (rotational) and curl-free (irrotational) components. The decomposition is exact: $v = u + \nabla p$ where $u$ is divergence-free and $\nabla p$ is curl-free.

In [ ]:
rng = np.random.default_rng(42)
v_x = gauss_filter(rng.standard_normal((n, n)), sigma=4)
v_y = gauss_filter(rng.standard_normal((n, n)), sigma=4)

# Decompose
u_x, u_y = proj_incompressible(v_x, v_y)   # divergence-free part
w_x, w_y = v_x - u_x, v_y - u_y           # curl-free part

print(f'|div(v)| = {np.std(div(v_x, v_y)):.4f}')
print(f'|div(u)| = {np.std(div(u_x, u_y)):.6f}   (div-free part, should be ~0)')
print(f'|curl(w)| = {np.std(np.gradient(w_y, axis=1) - np.gradient(w_x, axis=0)):.6f}  (curl-free part, should be ~0)')

x1d = np.linspace(0, 1, n)
X1, Y1 = np.meshgrid(x1d, x1d)
step = 4

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (vx, vy, title) in zip(axes, [
    (v_x, v_y, 'Original field $v$'),
    (u_x, u_y, 'Div-free part $u$'),
    (w_x, w_y, 'Curl-free part $\\nabla p$'),
]):
    speed = np.sqrt(vx**2 + vy**2)
    ax.streamplot(x1d, x1d, vx.T, vy.T, density=1.2, color=speed.T,
                  cmap='plasma', linewidth=1.5)
    ax.set_title(title, fontsize=10); ax.set_aspect('equal')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
fig.suptitle('Helmholtz–Hodge decomposition of a random smooth vector field', y=1.02)
plt.tight_layout()
plt.show()

## Semi-Lagrangian advection

Advecting a scalar field $\rho$ under a velocity field $u$ solves $\partial_t \rho + u \cdot \nabla \rho = 0$. The **semi-Lagrangian** scheme traces characteristics backward: $\rho^{t+\tau}(x) = \rho^t(x - \tau u(x))$.

In [ ]:
from scipy.ndimage import map_coordinates

ndef advect(rho, vx, vy, tau):
    """Semi-Lagrangian advection: rho(x) <- rho(x - tau*u(x))."""
    Y_c, X_c = np.mgrid[0:n, 0:n].astype(float)
    X_back = (X_c - tau * vx * n) % n
    Y_back = (Y_c - tau * vy * n) % n
    return map_coordinates(rho, [Y_back, X_back], order=3, mode='wrap')

# Divergence-free smooth velocity field
rng2 = np.random.default_rng(7)
vx0 = gauss_filter(rng2.standard_normal((n, n)), sigma=8)
vy0 = gauss_filter(rng2.standard_normal((n, n)), sigma=8)
vx0, vy0 = proj_incompressible(vx0, vy0)
norm_v = np.sqrt(vx0**2 + vy0**2).max()
vx0 /= norm_v; vy0 /= norm_v

# Initial density: Gaussian blob
xg = np.linspace(0, 1, n); yg = np.linspace(0, 1, n)
Xg, Yg = np.meshgrid(xg, yg)
rho0 = np.exp(-((Xg - 0.3)**2 + (Yg - 0.4)**2) / (2*0.07**2))

tau = 0.4
n_steps = 40
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
rho_t = rho0.copy()
for idx, ax in enumerate(axes.ravel()):
    if idx in [0, 5, 15, 25, 30, 35, 38, 39]:
        ax.imshow(rho_t, cmap='inferno', vmin=0, vmax=1, origin='upper')
        ax.set_title(f'step {idx}', fontsize=9); ax.axis('off')
    # advance
    for _ in range(max(1, idx - (idx-1 if idx > 0 else 0))):
        rho_t = advect(rho_t, vx0, vy0, tau)
        rho_t = np.clip(rho_t, 0, 1)

fig.suptitle('Semi-Lagrangian advection of a scalar field under incompressible flow', y=1.02)
plt.tight_layout()
plt.show()

## Viscous diffusion in Fourier space

The heat (diffusion) equation $\partial_t f = \nu \Delta f$ is solved **exactly** in Fourier space: $\hat{f}(k, t) = \hat{f}(k, 0) \cdot e^{-\nu \|k\|^2 t}$. High-frequency components are damped fastest, producing the characteristic smoothing effect of viscosity.

In [ ]:
def viscous_diffuse(f, nu, dt):
    F = np.fft.fft2(f)
    return np.fft.ifft2(F * np.exp(-nu * R2 * dt)).real

# Random sharp initial condition
f_sharp = rng.standard_normal((n, n))
n_vis_steps = [0, 1, 5, 20]
nu_vis = 0.5

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
f_curr = f_sharp.copy()
for ax, step in zip(axes, n_vis_steps):
    if step == 0:
        f_show = f_sharp
    else:
        f_show = viscous_diffuse(f_sharp, nu_vis, dt=step)
    ax.imshow(f_show, cmap='RdBu_r', vmin=-2, vmax=2)
    ax.set_title(f'$t = {step}\\nu$', fontsize=10); ax.axis('off')
fig.suptitle('Viscous diffusion: exact Fourier solution $\\hat{f}(k,t) = \\hat{f}_0 e^{-\\nu|k|^2 t}$', y=1.05)
plt.tight_layout()
plt.show()

## Interactive: vortex flow

We build a divergence-free velocity field from a stream function $\psi$ (so $u = \nabla^\perp \psi = (\partial_y \psi, -\partial_x \psi)$) and advect a dye pattern through it.

In [ ]:
def show_vortex(n_steps=20, nu=0.05):
    rng3 = np.random.default_rng(99)
    psi = gauss_filter(rng3.standard_normal((n, n)), sigma=10)
    # u = (d psi / dy, -d psi / dx)
    dpsi_x, dpsi_y = grad(psi)
    vx_v = dpsi_y
    vy_v = -dpsi_x
    speed_max = np.sqrt(vx_v**2 + vy_v**2).max()
    vx_v /= speed_max; vy_v /= speed_max

    # Initial dye: checkerboard
    xg = np.linspace(0, 1, n)
    Xg, Yg = np.meshgrid(xg, xg)
    rho_v = 0.5 + 0.5 * np.sign(np.sin(10*np.pi*Xg) * np.sin(10*np.pi*Yg))

    tau_v = 0.3
    for _ in range(n_steps):
        rho_v = advect(rho_v, vx_v, vy_v, tau_v)
        rho_v = viscous_diffuse(rho_v, nu, dt=1.0)
        rho_v = np.clip(rho_v, 0, 1)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    speed = np.sqrt(vx_v**2 + vy_v**2)
    axes[0].streamplot(xg, xg, vx_v.T, vy_v.T, density=1.5, color=speed.T,
                       cmap='plasma', linewidth=1.5)
    axes[0].set_title('Vortex velocity field'); axes[0].set_aspect('equal')
    axes[1].imshow(rho_v, cmap='inferno', vmin=0, vmax=1)
    axes[1].set_title(f'Dye after {n_steps} steps  ($\\nu={nu}$)')
    axes[1].axis('off')
    plt.tight_layout(); plt.show()

interact(show_vortex,
         n_steps=IntSlider(value=20, min=0, max=60, step=5, description='steps'),
         nu=FloatSlider(value=0.05, min=0.0, max=0.5, step=0.02, description='$\\nu$'));

## Bibliographical resources

- Chorin, A. J. (1968). Numerical solution of the Navier-Stokes equations. *Mathematics of Computation*, 22(104), 745–762.
- Stam, J. (1999). Stable fluids. *SIGGRAPH Proceedings*, 121–128.
- Canuto, C., Hussaini, M. Y., Quarteroni, A. and Zang, T. A. (2006). *Spectral Methods: Fundamentals in Single Domains*. Springer.
- Batchelor, G. K. (1967). *An Introduction to Fluid Dynamics*. Cambridge University Press.
- Bridson, R. (2015). *Fluid Simulation for Computer Graphics* (2nd ed.). CRC Press.